Accessing datetime column

In [55]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df=pd.read_csv('../solar_radiation_data/SolarPrediction.csv')

date_list=[]
for date in df['Data']:
    parts=date.split(' ')
    date_part=parts[0]
    date_list.append(date_part)

df['Date']=date_list

df['Datetime']=pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%m/%d/%Y %H:%M:%S')

new_df=df[['Datetime','Radiation','Temperature','Pressure','Humidity','WindDirection(Degrees)','Speed','TimeSunRise','TimeSunSet']]#merged datetime column
print(new_df.head())

             Datetime  Radiation  Temperature  Pressure  Humidity  \
0 2016-09-29 23:55:26       1.21           48     30.46        59   
1 2016-09-29 23:50:23       1.21           48     30.46        58   
2 2016-09-29 23:45:26       1.23           48     30.46        57   
3 2016-09-29 23:40:21       1.21           48     30.46        60   
4 2016-09-29 23:35:24       1.17           48     30.46        62   

   WindDirection(Degrees)  Speed TimeSunRise TimeSunSet  
0                  177.39   5.62    06:13:00   18:13:00  
1                  176.78   3.37    06:13:00   18:13:00  
2                  158.75   3.37    06:13:00   18:13:00  
3                  137.71   3.37    06:13:00   18:13:00  
4                  104.95   5.62    06:13:00   18:13:00  


Grouping columns by date and finding daily max , mean

In [56]:
daily_stats=df.groupby(df['Datetime'].dt.date)['Radiation'].agg(['max','mean'])  #grouping all dates by max and mean values of radiation
print(daily_stats)

                max        mean
Datetime                       
2016-09-01  1073.71  298.451600
2016-09-02  1062.01  333.471219
2016-09-03  1088.09  123.974574
2016-09-04  1601.26  153.584321
2016-09-05  1264.07  306.065125
...             ...         ...
2016-12-27   870.52  239.910556
2016-12-28  1157.27  174.910278
2016-12-29   696.30  125.018049
2016-12-30   439.43   88.367684
2016-12-31  1026.34  144.288542

[118 rows x 2 columns]


finding date of max radiation

In [57]:
max_date=daily_stats['max'].idxmax()  
print(f"Date with maximum radiation: {max_date}, Max Radiation: {daily_stats['max'].max()}")

Date with maximum radiation: 2016-09-04, Max Radiation: 1601.26


Time of max radiation that day, relating environmental data

In [58]:
filtered_row = df[df['Datetime'].dt.date == max_date]
#print(filtered_row)

peak_idx = filtered_row['Radiation'].idxmax()
peak_row = filtered_row.loc[peak_idx]

print('The time at which max radiation occurs is:', peak_row['Datetime'].time())      
print('The temperature at the time of max radiation is:', peak_row['Temperature'])
print('The humidity at the time of max radiation is:', peak_row['Humidity'])
print('The wind speed at the time of max radiation is:', peak_row['Speed'])

The time at which max radiation occurs is: 12:15:04
The temperature at the time of max radiation is: 61
The humidity at the time of max radiation is: 93
The wind speed at the time of max radiation is: 9.0


Time difference between sunsrise and peak radiation

In [59]:
sunrise_time=peak_row['TimeSunRise']
sunset_time=peak_row['TimeSunSet']
print(f'Sunrise happend at **{sunrise_time}** and Sunset happened at **{sunset_time}** on the day of max radiation.')
print(f'Whereas the time of max radiation was at **{peak_row["Datetime"].time()}**')
print()

sunrise_datetime=pd.to_datetime(peak_row['Datetime'].date().strftime('%Y-%m-%d') + ' ' + peak_row['TimeSunRise'])
time_difference=(peak_row['Datetime'] - sunrise_datetime)
a=time_difference.components

print('Time taken for radiation to reach its peak from sunrise is:')
print(f"{a.hours} hours, {a.minutes} minutes, and {a.seconds} seconds.")

Sunrise happend at **06:08:00** and Sunset happened at **18:35:00** on the day of max radiation.
Whereas the time of max radiation was at **12:15:04**

Time taken for radiation to reach its peak from sunrise is:
6 hours, 7 minutes, and 4 seconds.


Top 5 days with highest average radiation

In [ ]:
top_5_mean=daily_stats.nlargest(5, 'mean')
print("Top 5 dates with highest mean radiation:")
i=1
for value in top_5_mean.index:
    print(f"{i}. Date: {value}, Mean Radiation: {daily_stats.loc[value, 'mean'].round(3)}")
    i += 1  

Top 5 dates with highest mean radiation:
1. Date: 2016-09-19, Mean Radiation: 335.504
2. Date: 2016-09-02, Mean Radiation: 333.471
3. Date: 2016-09-22, Mean Radiation: 325.516
4. Date: 2016-09-25, Mean Radiation: 319.922
5. Date: 2016-09-28, Mean Radiation: 312.4
